In [30]:
import pandas as pd

df_atletas = pd.read_csv('../data/athlete_events.csv')
df_regiones = pd.read_csv('../data/noc_regions.csv')

df_atletas['IMC'] = df_atletas['Weight'] / ((df_atletas['Height'] / 100) ** 2)
df_atletas = pd.merge(df_atletas, df_regiones, on='NOC', how='left')
df_atletas['medalla'] = df_atletas['Medal'].notna().astype(int)

### 11. Clasificación de atletas por volumen de pruebas (n_pruebas)

In [31]:
df_atletas['n_pruebas'] = df_atletas.groupby(['ID', 'Games'])['Event'].transform('count')

df_atletas['categoria_pruebas'] = pd.cut(
    df_atletas['n_pruebas'],
    bins=[0, 1, 3, df_atletas['n_pruebas'].max()],
    labels=['1 prueba', '2-3 pruebas', '4 o más pruebas'])

df_atletas['id_edicion'] = df_atletas['ID'].astype(str) + '_' + df_atletas['Games']

tabla_pruebas = df_atletas.groupby('categoria_pruebas', observed=False).agg(
    participaciones=('ID', 'count'),
    atletas_edicion=('id_edicion', 'nunique'),
    medallas=('medalla', 'sum'),
    tasa_medalla_pct=('medalla', lambda x: round(x.mean() * 100, 2)))

display(tabla_pruebas)

deportes_por_grupo = pd.crosstab(df_atletas['Sport'], df_atletas['categoria_pruebas'], normalize='columns').mul(100).round(2)
for grupo in deportes_por_grupo.columns:
    display(deportes_por_grupo[grupo].sort_values(ascending=False).head(5))

tabla_pruebas.to_csv('../outputs/tablas/medallas_por_n_pruebas.csv')

,participaciones,atletas_edicion,medallas,tasa_medalla_pct
categoria_pruebas,,,,
1 prueba,139986,139986,24464,17.48
2-3 pruebas,88160,39774,11281,12.80
4 o más pruebas,42970,7692,4038,9.40


Sport
Athletics    17.07
Rowing        7.05
Football      4.81
Wrestling     4.68
Sailing       4.50
Name: 1 prueba, dtype: float64

Sport
Athletics        15.68
Swimming         15.49
Fencing           7.77
Alpine Skiing     7.11
Cycling           6.35
Name: 2-3 pruebas, dtype: float64

Sport
Gymnastics              57.52
Swimming                10.43
Cross Country Skiing     8.22
Biathlon                 4.06
Shooting                 3.86
Name: 4 o más pruebas, dtype: float64

### Análisis según el volumen de participación (n_pruebas)

- **Distribución:** la mayoría de los atletas compite en una sola prueba por edición (139.986 de 187.452 atletas-edición). Los de 2 a 3 pruebas son 39.774 y los de 4 o más solo 7.692, aunque este último grupo acumula 42.970 participaciones.
- **Medallas:** la tasa de medalla baja al aumentar las pruebas: 17,48 % con una prueba, 12,80 % con 2 a 3 y 9,40 % con 4 o más.
- **Deportes:** en el grupo de una prueba domina el atletismo (17,07 %); en el de 2 a 3, atletismo (15,68 %) y natación (15,49 %); en el de 4 o más, la gimnasia (57,52 %).
- **Tamaño de los grupos:** ningún grupo queda vacío, pero el de 4 o más pruebas es pequeño (7.692) y está dominado por un solo deporte.

### 12. Análisis por grupo etario (grupo_etario) y proporción de medallas

In [32]:
df_atletas['grupo_etario'] = pd.cut(
    df_atletas['Age'],
    bins=[0, 17, 34, df_atletas['Age'].max()],
    labels=['Menores', 'Jóvenes adultos', 'Mayores'])
df_atletas['grupo_etario'] = df_atletas['grupo_etario'].cat.add_categories('Sin edad').fillna('Sin edad')

df_atletas['con_talla_peso'] = (df_atletas['Height'].notna() & df_atletas['Weight'].notna()).astype(int)

tabla_edad = df_atletas.groupby(['grupo_etario', 'Sex'], observed=True).agg(
    registros=('ID', 'count'),
    pct_con_talla_peso=('con_talla_peso', lambda x: round(x.mean() * 100, 2)),
    pct_sin_talla_peso=('con_talla_peso', lambda x: round((1 - x.mean()) * 100, 2)),
    pct_con_medalla=('medalla', lambda x: round(x.mean() * 100, 2)),
    pct_sin_medalla=('medalla', lambda x: round((1 - x.mean()) * 100, 2)))

display(tabla_edad)

tabla_edad.to_csv('../outputs/tablas/grupo_etario_sexo.csv')

registros  pct_con_talla_peso  pct_sin_talla_peso  \
grupo_etario    Sex                                                      
Menores         F         9336               86.21               13.79   
                M         3172               70.65               29.35   
Jóvenes adultos F        61889               90.90                9.10   
                M       167728               76.86               23.14   
Mayores         F         2873               83.78               16.22   
                M        16644               49.88               50.12   
Sin edad        F          424                4.25               95.75   
                M         9050                7.40               92.60   

                     pct_con_medalla  pct_sin_medalla  
grupo_etario    Sex                                    
Menores         F               9.50            90.50  
                M               9.21            90.79  
Jóvenes adultos F              16.08            83.92  
                M              14.97            85.03  
Mayores         F              13.85            86.15  
                M              14.47            85.53  
Sin edad        F               4.01            95.99  
                M               7.90            92.10

### Análisis por grupo etario y sexo

- **Sin edad:** los atletas sin edad registrada (9.474) no se eliminaron; se agruparon en la categoría "Sin edad".
- **Participación:** los jóvenes adultos concentran la mayoría de los registros, y los hombres superan a las mujeres en todos los grupos salvo los menores.
- **Medallas:** la tasa es más baja en los menores (9,50 % mujeres, 9,21 % hombres) y más alta en los jóvenes adultos (16,08 % mujeres, 14,97 % hombres). En los mayores es de 13,85 % en mujeres y 14,47 % en hombres.
- **Estatura y peso:** faltan más en los hombres que en las mujeres en todos los grupos. La diferencia mayor está en los mayores: 50,12 % de los hombres sin datos frente a 16,22 % de las mujeres.
- **Sin edad:** este grupo casi no tiene estatura ni peso (solo 4,25 % de mujeres y 7,40 % de hombres) y tiene las tasas de medalla más bajas (4,01 % y 7,90 %).
- **Conclusión:** los jóvenes adultos tienen la mayor proporción de medallas y los menores la menor. Las diferencias por sexo en medallas son pequeñas, pero en datos faltantes son grandes.